# Проверка гипотезы по ФП группы 28 (МкБ/МСБ) - вариант 26.2/26.2У

Гипотеза: факторы группы `28*` обычно возникают после факторов групп `14*`, `15*`, `16*`, `24*`, `26*`.

Когорты:
1. ИНН, у которых **первый ФП** (в выборке) возник в 2024 году.
2. ИНН, у которых **первый ФП группы 28** (в выборке) возник в 2024 году.

Особое правило этого варианта:
- в группе предшественников `26` учитываются **только** коды `26.2` и `26.2У`;
- остальные коды `26.x` в группу `26` не включаются.

База готовится по логике `analysis_crm_segments.ipynb` (1-в-1), затем ограничивается сегментами `МкБ`/`МСБ` и типом `ФП`.

In [ ]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", None)

# Пути к файлам
DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"

# Период анализа
DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO = pd.Timestamp("2025-12-31")

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}
ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]

TARGET_SEGMENTS = ["МкБ", "МСБ"]
FP_TYPE = "ФП"
PRIOR_GROUPS = ["14", "15", "16", "24", "26"]
TARGET_GROUP = "28"
ALLOWED_26_CODES = {"26.2", "26.2У"}


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    if len(s) <= 10:
        return s.zfill(10)
    return s.zfill(12)


def normalize_code_for_grouping(code: str) -> str:
    if pd.isna(code):
        return ""
    code = str(code).strip().replace(" ", "")
    # Выравниваем разные варианты буквы У в суффиксе кода.
    code = code.upper().replace("Y", "У").replace("U", "У")
    return code


def detect_prior_group(code: str):
    norm = normalize_code_for_grouping(code)

    if norm.startswith("14"):
        return "14"
    if norm.startswith("15"):
        return "15"
    if norm.startswith("16"):
        return "16"
    if norm.startswith("24"):
        return "24"
    if norm in ALLOWED_26_CODES:
        return "26"

    return None


COHORT_LABELS = {
    "cohort1_first_fp_2024": "Когорта 1: первый ФП в 2024",
    "cohort2_first_28_2024": "Когорта 2: первый ФП группы 28* в 2024",
}

SUMMARY_COLS_RU_EVENT = {
    "cohort": "Когорта",
    "segment": "Сегмент",
    "cohort_inn": "ИНН в когорте",
    "events_28": "Количество событий 28*",
    "inn_with_28": "Количество ИНН с 28*",
    "events_any_prior_before_28": "События 28* с предшественниками 14*/15*/16*/24*/26*",
    "inn_with_any_prior_before_28": "ИНН с предшественниками 14*/15*/16*/24*/26* до 28*",
    "events_any_other_fp_before_28": "События 28* с любым другим ФП (кроме 28*) до 28*",
    "inn_with_any_other_fp_before_28": "ИНН с любым другим ФП (кроме 28*) до 28*",
    "prior_events_any_other_fp_before_28": "Количество событий других ФП (кроме 28*) до 28*",
    "share_inn_with_28": "Доля ИНН с 28* в когорте",
    "share_events_any_prior_before_28": "Доля событий 28* с предшественниками 14*/15*/16*/24*/26*",
    "share_inn_any_prior_before_28": "Доля ИНН с предшественниками 14*/15*/16*/24*/26* до 28*",
    "share_events_any_other_fp_before_28": "Доля событий 28* с любым другим ФП (кроме 28*) до 28*",
    "share_inn_any_other_fp_before_28": "Доля ИНН с любым другим ФП (кроме 28*) до 28*",
    "avg_fp_before_28_any_other_fp": "Среднее число других ФП (кроме 28*) до 28* на событие 28*",
}

BY_GROUP_COLS_RU_EVENT = {
    "cohort": "Когорта",
    "segment": "Сегмент",
    "prior_group": "Группа предшественника",
    "prior_events_before_28": "Количество событий группы до 28*",
    "event_28_with_group_before_28": "События 28* с группой-предшественником",
    "inn_with_group_before_28": "ИНН с группой-предшественником до 28*",
    "events_28": "Количество событий 28*",
    "inn_with_28": "Количество ИНН с 28*",
    "share_events_with_group_before_28": "Доля событий 28* с группой-предшественником",
    "share_inn_with_group_before_28": "Доля ИНН с группой-предшественником до 28*",
}

TOP5_COLS_RU = {
    "cohort": "Когорта",
    "segment": "Сегмент",
    "rank": "Ранг",
    "pred_code": "Код ФП-предшественника",
    "prior_event_count": "Количество срабатываний предшественника до 28*",
    "event_28_count": "Количество событий 28* с этим предшественником",
    "inn_count": "Количество ИНН с этим предшественником",
}


def add_readable_labels(df: pd.DataFrame, cohort_col: str = "cohort", prior_group_col: str = "prior_group") -> pd.DataFrame:
    out = df.copy()
    if cohort_col in out.columns:
        out[cohort_col] = out[cohort_col].astype(str).map(COHORT_LABELS).fillna(out[cohort_col].astype(str))
    if prior_group_col in out.columns:
        out[prior_group_col] = out[prior_group_col].astype(str).apply(lambda x: f"{x}*" if x != "nan" else x)
    return out


print("Параметры заданы.")
print(f"Период: {DATE_FROM:%d.%m.%Y} - {DATE_TO:%d.%m.%Y}")
print("Ограничение для группы 26: только коды 26.2 и 26.2У")

In [ ]:
# Подготовка данных: 1-в-1 с analysis_crm_segments.ipynb

df = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
print(f"Загружено: {len(df):,} строк")

df["inn"] = df["X_INN"].apply(normalize_inn)
df["dt"] = pd.to_datetime(df["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")

df = df[(df["dt"] >= DATE_FROM) & (df["dt"] <= DATE_TO)].copy()
print(f"После фильтра {DATE_FROM:%d.%m.%Y} - {DATE_TO:%d.%m.%Y}: {len(df):,} строк")

if "VAL" in df.columns:
    before_val = len(df)
    df = df[df["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()
    print(f"Фильтр по источникам ({', '.join(ALLOWED_SOURCES)}): {len(df):,} строк (удалено {before_val - len(df):,})")

df["TYPE_FP"] = df["TYPE_FP"].str.strip().replace({"FP": "ФП", "SFP": "СФП"})

df["X_AREA_RESP"] = df["X_AREA_RESP"].astype(str).str.strip()
df["segment"] = df["X_AREA_RESP"].map(SEGMENT_MAP)

unmapped = df[df["segment"].isna()]["X_AREA_RESP"].value_counts()
if len(unmapped) > 0:
    print("\nНемаппированные значения X_AREA_RESP (будут исключены):")
    print(unmapped.to_string())

df = df[df["segment"].notna()].copy()
print(f"После маппинга сегментов: {len(df):,} строк")

df["fp_num"] = df["NUMBER_FP_SFP"].astype(str).str.strip()

before_drop = len(df)
df = df.dropna(subset=["inn", "NUMBER_FP_SFP"]).copy()
print(f"После dropna(inn, NUMBER_FP_SFP): {len(df):,} строк (удалено {before_drop - len(df):,})")

before_dedup = len(df)
df = df.drop_duplicates(subset=["inn", "fp_num", "TYPE_FP", "IDENTIFICATION_DATE"]).copy()
print(f"После дедупликации (INN + fp_num + TYPE_FP + DATE): {len(df):,} строк (удалено {before_dedup - len(df):,} дублей)")

# Ограничение задачи: только МкБ/МСБ и только ФП.
df_fp = df[(df["segment"].isin(TARGET_SEGMENTS)) & (df["TYPE_FP"] == FP_TYPE)].copy()
print("\nПосле ограничения на МкБ/МСБ + TYPE_FP='ФП':")
print(f"  Строк: {len(df_fp):,}")
print(f"  Уникальных ИНН: {df_fp['inn'].nunique():,}")
print(df_fp.groupby("segment")["inn"].nunique().rename("inn_nunique").to_string())

In [ ]:
# Построение когорт

# Когорта 1: первый ФП (в выборке) в 2024.
first_fp_by_inn_seg = (
    df_fp.groupby(["segment", "inn"], as_index=False)
    .agg(first_fp_dt=("dt", "min"))
)
cohort1 = first_fp_by_inn_seg[first_fp_by_inn_seg["first_fp_dt"].dt.year == 2024][["segment", "inn"]].copy()
cohort1["cohort"] = "cohort1_first_fp_2024"

# Когорта 2: первый 28* ФП (в выборке) в 2024.
first_28_by_inn_seg = (
    df_fp[df_fp["fp_num"].astype(str).str.startswith(TARGET_GROUP)]
    .groupby(["segment", "inn"], as_index=False)
    .agg(first_28_dt=("dt", "min"))
)
cohort2 = first_28_by_inn_seg[first_28_by_inn_seg["first_28_dt"].dt.year == 2024][["segment", "inn"]].copy()
cohort2["cohort"] = "cohort2_first_28_2024"

print("Размеры когорт (ИНН):")
print(
    pd.concat([
        cohort1.groupby("segment")["inn"].nunique().rename("cohort1_inn"),
        cohort2.groupby("segment")["inn"].nunique().rename("cohort2_inn"),
    ], axis=1).fillna(0).astype(int).to_string()
)

In [ ]:
# Новая событийная логика
# Предшественники считаются до КАЖДОГО события 28*.


def compute_event_based_metrics_for_cohort(cohort_df: pd.DataFrame, cohort_name: str):
    c = cohort_df[["segment", "inn"]].drop_duplicates().copy()
    c["cohort"] = cohort_name

    cohort_events = df_fp.merge(c[["segment", "inn"]], on=["segment", "inn"], how="inner")

    # Все события 28* в когорте (каждое событие - отдельный anchor).
    events_28 = cohort_events[cohort_events["fp_num"].astype(str).str.startswith(TARGET_GROUP)].copy()
    events_28 = events_28[["segment", "inn", "dt", "fp_num"]].rename(
        columns={"dt": "dt_28", "fp_num": "fp28_code"}
    )
    events_28 = events_28.reset_index(drop=True)
    events_28["event_28_id"] = events_28.index.astype(int)

    # Частоты 28
    events_28_stats = (
        events_28.groupby("segment", as_index=False)
        .agg(events_28=("event_28_id", "nunique"), inn_with_28=("inn", "nunique"))
    )

    # Все ФП до каждого события 28* (строго раньше даты конкретного события 28).
    prior_all = (
        events_28[["event_28_id", "segment", "inn", "dt_28"]]
        .merge(
            cohort_events[["segment", "inn", "dt", "fp_num"]].rename(columns={"dt": "prior_dt", "fp_num": "prior_code"}),
            on=["segment", "inn"],
            how="left",
        )
    )
    prior_all = prior_all[prior_all["prior_dt"] < prior_all["dt_28"]].copy()

    # Исключаем из предшественников саму группу 28*.
    prior_all = prior_all[~prior_all["prior_code"].astype(str).str.startswith(TARGET_GROUP)].copy()

    # Гипотезные группы 14/15/16/24/26.
    # Для группы 26 в этом варианте включаются только 26.2 и 26.2У.
    prior_all["prior_group"] = prior_all["prior_code"].apply(detect_prior_group)

    # Только гипотезные группы.
    prior_h = prior_all[prior_all["prior_group"].notna()].copy()

    # Уникальные event_28, у которых был хотя бы один prior из 14/15/16/24/26.
    events_with_any_prior = (
        prior_h[["segment", "event_28_id"]]
        .drop_duplicates()
        .groupby("segment", as_index=False)
        .agg(events_any_prior_before_28=("event_28_id", "nunique"))
    )

    # Уникальные ИНН с any_prior из 14/15/16/24/26 до хотя бы одного события 28.
    inn_with_any_prior = (
        prior_h[["segment", "inn"]]
        .drop_duplicates()
        .groupby("segment", as_index=False)
        .agg(inn_with_any_prior_before_28=("inn", "nunique"))
    )

    # Метрики по любым другим ФП (кроме 28*)
    events_with_any_other = (
        prior_all[["segment", "event_28_id"]]
        .drop_duplicates()
        .groupby("segment", as_index=False)
        .agg(events_any_other_fp_before_28=("event_28_id", "nunique"))
    )
    inn_with_any_other = (
        prior_all[["segment", "inn"]]
        .drop_duplicates()
        .groupby("segment", as_index=False)
        .agg(inn_with_any_other_fp_before_28=("inn", "nunique"))
    )
    prior_events_any_other = (
        prior_all.groupby("segment", as_index=False)
        .agg(prior_events_any_other_fp_before_28=("prior_code", "size"))
    )

    cohort_size = c.groupby("segment", as_index=False).agg(cohort_inn=("inn", "nunique"))

    summary = (
        cohort_size
        .merge(events_28_stats, on="segment", how="left")
        .merge(events_with_any_prior, on="segment", how="left")
        .merge(inn_with_any_prior, on="segment", how="left")
        .merge(events_with_any_other, on="segment", how="left")
        .merge(inn_with_any_other, on="segment", how="left")
        .merge(prior_events_any_other, on="segment", how="left")
    )

    for col in [
        "events_28", "inn_with_28", "events_any_prior_before_28", "inn_with_any_prior_before_28",
        "events_any_other_fp_before_28", "inn_with_any_other_fp_before_28", "prior_events_any_other_fp_before_28"
    ]:
        summary[col] = summary[col].fillna(0).astype(int)

    summary["share_inn_with_28"] = np.where(summary["cohort_inn"] > 0, summary["inn_with_28"] / summary["cohort_inn"], np.nan)
    summary["share_events_any_prior_before_28"] = np.where(summary["events_28"] > 0, summary["events_any_prior_before_28"] / summary["events_28"], np.nan)
    summary["share_inn_any_prior_before_28"] = np.where(summary["inn_with_28"] > 0, summary["inn_with_any_prior_before_28"] / summary["inn_with_28"], np.nan)

    summary["share_events_any_other_fp_before_28"] = np.where(
        summary["events_28"] > 0,
        summary["events_any_other_fp_before_28"] / summary["events_28"],
        np.nan,
    )
    summary["share_inn_any_other_fp_before_28"] = np.where(
        summary["inn_with_28"] > 0,
        summary["inn_with_any_other_fp_before_28"] / summary["inn_with_28"],
        np.nan,
    )
    summary["avg_fp_before_28_any_other_fp"] = np.where(
        summary["events_28"] > 0,
        summary["prior_events_any_other_fp_before_28"] / summary["events_28"],
        np.nan,
    )

    # Разбивка по группам 14/15/16/24/26
    by_group = (
        prior_h.groupby(["segment", "prior_group"], as_index=False)
        .agg(
            prior_events_before_28=("prior_code", "size"),
            event_28_with_group_before_28=("event_28_id", "nunique"),
            inn_with_group_before_28=("inn", "nunique"),
        )
    )

    by_group = by_group.merge(events_28_stats[["segment", "events_28", "inn_with_28"]], on="segment", how="left")
    by_group["share_events_with_group_before_28"] = np.where(
        by_group["events_28"] > 0,
        by_group["event_28_with_group_before_28"] / by_group["events_28"],
        np.nan,
    )
    by_group["share_inn_with_group_before_28"] = np.where(
        by_group["inn_with_28"] > 0,
        by_group["inn_with_group_before_28"] / by_group["inn_with_28"],
        np.nan,
    )

    # TOP-5 любых предшествующих ФП (кроме 28*) по частоте.
    top_base = (
        prior_all.groupby(["segment", "prior_code"], as_index=False)
        .agg(
            prior_event_count=("prior_code", "size"),
            event_28_count=("event_28_id", "nunique"),
            inn_count=("inn", "nunique"),
        )
        .rename(columns={"prior_code": "pred_code"})
        .sort_values(
            ["segment", "prior_event_count", "event_28_count", "inn_count", "pred_code"],
            ascending=[True, False, False, False, True],
        )
    )
    top_base["rank"] = top_base.groupby("segment").cumcount() + 1
    top5 = top_base[top_base["rank"] <= 5].copy()
    top5 = top5[["segment", "rank", "pred_code", "prior_event_count", "event_28_count", "inn_count"]]

    summary.insert(0, "cohort", cohort_name)
    by_group.insert(0, "cohort", cohort_name)
    top5.insert(0, "cohort", cohort_name)

    # Нулевые строки по сегментам/группам для стабильного вывода.
    seg_group_base = pd.MultiIndex.from_product([TARGET_SEGMENTS, PRIOR_GROUPS], names=["segment", "prior_group"]).to_frame(index=False)
    by_group = seg_group_base.merge(by_group, on=["segment", "prior_group"], how="left")
    by_group["cohort"] = by_group["cohort"].fillna(cohort_name)
    for col in ["prior_events_before_28", "event_28_with_group_before_28", "inn_with_group_before_28", "events_28", "inn_with_28"]:
        by_group[col] = by_group[col].fillna(0).astype(int)
    by_group["share_events_with_group_before_28"] = np.where(by_group["events_28"] > 0, by_group["event_28_with_group_before_28"] / by_group["events_28"], np.nan)
    by_group["share_inn_with_group_before_28"] = np.where(by_group["inn_with_28"] > 0, by_group["inn_with_group_before_28"] / by_group["inn_with_28"], np.nan)

    return summary, by_group, top5


event_summary1, event_group1, top5_1 = compute_event_based_metrics_for_cohort(cohort1, "cohort1_first_fp_2024")
event_summary2, event_group2, top5_2 = compute_event_based_metrics_for_cohort(cohort2, "cohort2_first_28_2024")

cohort_segment_summary_event = pd.concat([event_summary1, event_summary2], ignore_index=True)
before_28_by_group_event = pd.concat([event_group1, event_group2], ignore_index=True)
top5_prior_fp_by_cohort_segment = pd.concat([top5_1, top5_2], ignore_index=True)

cohort_order = ["cohort1_first_fp_2024", "cohort2_first_28_2024"]
segment_order = ["МкБ", "МСБ"]

for d in [cohort_segment_summary_event, before_28_by_group_event, top5_prior_fp_by_cohort_segment]:
    d["cohort"] = pd.Categorical(d["cohort"], categories=cohort_order, ordered=True)
    d["segment"] = pd.Categorical(d["segment"], categories=segment_order, ordered=True)

cohort_segment_summary_event = cohort_segment_summary_event.sort_values(["cohort", "segment"]).reset_index(drop=True)
before_28_by_group_event = before_28_by_group_event.sort_values(["cohort", "segment", "prior_group"]).reset_index(drop=True)
top5_prior_fp_by_cohort_segment = top5_prior_fp_by_cohort_segment.sort_values(["cohort", "segment", "rank"]).reset_index(drop=True)

# Понятные русские названия для вывода и выгрузки.
cohort_segment_summary_event_ru = add_readable_labels(cohort_segment_summary_event)
before_28_by_group_event_ru = add_readable_labels(before_28_by_group_event)
top5_prior_fp_by_cohort_segment_ru = add_readable_labels(top5_prior_fp_by_cohort_segment)

cohort_segment_summary_event_ru = cohort_segment_summary_event_ru.rename(columns=SUMMARY_COLS_RU_EVENT)
before_28_by_group_event_ru = before_28_by_group_event_ru.rename(columns=BY_GROUP_COLS_RU_EVENT)
top5_prior_fp_by_cohort_segment_ru = top5_prior_fp_by_cohort_segment_ru.rename(columns=TOP5_COLS_RU)

print("Сводная таблица по когортам (событийная логика, 26 = только 26.2 и 26.2У)")
display(cohort_segment_summary_event_ru)

print("\nТаблица по группам-предшественникам (событийная логика)")
display(before_28_by_group_event_ru)

print("\nTOP-5 предшественников (событийная логика)")
display(top5_prior_fp_by_cohort_segment_ru)

In [ ]:
# Экспорт новой событийной логики
OUT_XLSX = Path(DATA_DIR) / "fp28_hypothesis_cohorts_event_based_26_2_only.xlsx"

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    cohort_segment_summary_event_ru.to_excel(writer, sheet_name="cohort_segment_summary", index=False)
    before_28_by_group_event_ru.to_excel(writer, sheet_name="before_28_by_group", index=False)
    top5_prior_fp_by_cohort_segment_ru.to_excel(writer, sheet_name="top5_prior_fp", index=False)

print(f"Сохранено: {OUT_XLSX}")